# English-Luganda Translator - ML Pipeline on Colab

**GPU-Accelerated Machine Learning Workflow**

This notebook trains an English-Luganda translator using:
- Week 2: ML Workflow (load → preprocess → train → evaluate)
- Week 3: Regularization (dropout, L2)
- Week 6: Evaluation Metrics (BLEU score)
- Week 9: Transformers (sequence-to-sequence)

**Datasets**: 5 parallel corpora (3100+ pairs)
**GPU**: Free Tesla T4 on Colab (5-10x faster than CPU)
**Time**: ~15-20 minutes total

## STEP 1: Setup Environment

Install required packages

In [ ]:
import subprocess
import sys

print('='*80)
print('ENGLISH-LUGANDA TRANSLATOR - COLAB ML PIPELINE')
print('='*80)
print('\\n[STEP 1: Installing packages]\\n')

packages = [
    'torch',
    'transformers',
    'datasets',
    'pandas',
    'scikit-learn',
    'sacrebleu',
]

print('📦 Installing packages...')
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
    print(f'  ✓ {package}')

print('\\n✅ All packages installed')

## STEP 2: Mount Google Drive & Setup Project

Connect to your Google Drive where the project is uploaded

In [ ]:
from google.colab import drive
import os
from pathlib import Path

print('\\n[STEP 2: Setting up project]\\n')

# Mount Google Drive
print('📁 Mounting Google Drive...')
drive.mount('/content/drive')
print('✅ Drive mounted\\n')

# Set working directory
COLAB_PROJECT_PATH = '/content/drive/My Drive/English-Luganda-Translator/ENGLISH-LUGANDA-TRANSLATOR'

if not os.path.exists(COLAB_PROJECT_PATH):
    print(f'⚠️  Path not found: {COLAB_PROJECT_PATH}')
    print('Update COLAB_PROJECT_PATH based on your folder location')
else:
    os.chdir(COLAB_PROJECT_PATH)
    print(f'✅ Working directory: {os.getcwd()}')

## STEP 3: Load All Datasets

Load and combine 5 English-Luganda parallel datasets

In [ ]:
print('\\n[STEP 3: Loading datasets]\\n')
print('='*80)

import sys
sys.path.insert(0, 'src')

from load_data import load_all_datasets, get_dataset_statistics
from utils import print_section

combined_df = load_all_datasets()
stats = get_dataset_statistics(combined_df)

print(f'\\n✅ Dataset Statistics:')
print(f'   Total samples: {stats["total_samples"]:,}')
print(f'   English text - Avg length: {stats["avg_english_length"]:.1f}')
print(f'   Luganda text - Avg length: {stats["avg_luganda_length"]:.1f}')

## STEP 4: Preprocess & Create Splits

Clean text and create train (80%) / val (10%) / test (10%) splits

In [ ]:
print('\\n[STEP 4: Preprocessing & creating splits]\\n')
print('='*80)

from preprocess import preprocess_and_split, save_splits

train_df, val_df, test_df = preprocess_and_split(combined_df)
save_splits(train_df, val_df, test_df)

print(f'\\n✅ Data splits created:')
print(f'   Train: {len(train_df):,}')
print(f'   Val: {len(val_df):,}')
print(f'   Test: {len(test_df):,}')

## STEP 5: Train Model on GPU

⚠️ **This will take 8-12 minutes on GPU**

Fine-tune transformer model with your data

In [ ]:
print('\\n[STEP 5: Training model]\\n')
print('⚠️  This will take 8-12 minutes on GPU\\n')

import torch

print(f'🤖 GPU Status:')
print(f'   Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   Device: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\\n' + '='*80)
print('STARTING TRAINING')
print('='*80 + '\\n')

from train import main as train_main

try:
    model, tokenizer = train_main()
    print(f'\\n✅ Training complete!')
except Exception as e:
    print(f'❌ Training error: {e}')

## STEP 6: Evaluate Model

Calculate BLEU score on test set

In [ ]:
print('\\n[STEP 6: Evaluating model]\\n')
print('='*80)

from evaluate import main as eval_main

try:
    eval_main()
    print(f'\\n✅ Evaluation complete!')
except Exception as e:
    print(f'❌ Evaluation error: {e}')

## STEP 7: Display Results

View BLEU score and metrics

In [ ]:
print('\\n[STEP 7: Results Summary]\\n')

import json
from pathlib import Path

eval_file = Path('outputs/evaluation_results.json')
if eval_file.exists():
    with open(eval_file) as f:
        results = json.load(f)
    
    print('\\n' + '='*80)
    print('📊 FINAL RESULTS')
    print('='*80)
    print(f'\\n✅ BLEU Score: {results["bleu_score"]:.2f}')
    print(f'   Test samples: {results["num_test_samples"]}')
    print(f'   Avg prediction length: {results["avg_prediction_length"]:.1f}')
    print(f'   Avg reference length: {results["avg_reference_length"]:.1f}')

## STEP 8: Download Results & Model

Download trained model and evaluation results

In [ ]:
print('\\n[STEP 8: Downloading files]\\n')

from google.colab import files
import shutil

print('📥 Preparing files for download...\\n')

print('   Zipping trained model...')
shutil.make_archive('trained_model', 'zip', 'models')

print('   Zipping evaluation results...')
shutil.make_archive('evaluation_outputs', 'zip', 'outputs')

print('\\n📥 Download files:')
print('   1. trained_model.zip - Use for inference')
print('   2. evaluation_outputs.zip - BLEU scores and predictions')

files.download('trained_model.zip')
files.download('evaluation_outputs.zip')

print('\\n✅ Files downloaded!')

## ✅ Pipeline Complete!

### Summary
- ✓ Loaded all 5 datasets
- ✓ Created train/val/test splits
- ✓ Trained model on GPU
- ✓ Evaluated on test set
- ✓ Downloaded results

### Next Steps
1. Download the trained model
2. Extract and use locally
3. Fine-tune further if needed
4. Deploy to production